# 01 — Canonical dataset and base-model baseline

This notebook validates the canonical dataset, creates the fixed 72/8 split, loads the untouched Qwen2.5-VL base model in 4-bit form, and evaluates exactly eight validation sequences. No training occurs here.

In [6]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / "Scripts" / "dataUtils.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate the repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Scripts import dataUtils as dU
from Scripts import trainValSplit
from Scripts.evaluator import calculate_metrics, evaluate_model, inspect_prediction, load_prediction_records, display_metric_summary, prediction_report

SEED = 42
MODEL_PATH = Path(os.environ.get("MANGA_VLM_MODEL_PATH", r"C:\models\HF\Qwen2.5-VL-3B-Instruct"))
LABELS_PATH = ROOT / "TaskData" / "development" / "labels.jsonl"
IMAGE_ROOT = ROOT / "TaskData" / "development" / "images"
SPLIT_PATH = ROOT / "outputs" / "canonical_split_seed42.json"
OUTPUT_DIR = ROOT / "outputs" / "canonical_base_baseline_better_prompt"
MAX_NEW_TOKENS = 1024
MINIMUM_SIMILARITY = 0.50
REPETITION_PENALTY = 1.05

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [7]:
labels = dU.load_labels(LABELS_PATH)
all_ids = list(labels)
train_ids, validation_ids = trainValSplit.fit_seq(all_ids, v_size=0.10, random_state=SEED)
split = {"seed": SEED, "validation_fraction": 0.10, "train_ids": train_ids, "validation_ids": validation_ids}
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
if SPLIT_PATH.is_file():
    existing_split = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
    if existing_split != split:
        raise RuntimeError(f"Existing split differs from the seed-42 split: {SPLIT_PATH}")
else:
    SPLIT_PATH.write_text(json.dumps(split, indent=2), encoding="utf-8")

validation_dataset = dU.MangaSequenceDataset(LABELS_PATH, IMAGE_ROOT, validation_ids)
assert len(all_ids) == 80
assert len(train_ids) == 72 and len(validation_dataset) == 8
assert set(train_ids).isdisjoint(validation_ids)
display(pd.Series({"all sequences": len(all_ids), "training": len(train_ids), "validation": len(validation_dataset)}, name="count").to_frame())
display(pd.DataFrame({"validation_sequence_id": validation_ids}))

,count
all sequences,80
training,72
validation,8


,validation_sequence_id
0,seq_d8bc3c6ac1bb6017
1,seq_2032620aa4e4ac7f
2,seq_d7af6c64ebc22bbe
3,seq_535e0754d805105b
4,seq_210721b0d576149f
5,seq_dce8be23c3281228
6,seq_1c1d4f5e71a2f554
7,seq_098e260f231aa789


In [8]:
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Local model not found: {MODEL_PATH}")
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True, min_pixels=128 * 28 * 28, max_pixels=256 * 28 * 28)
prompt_builder = dU.MangaPromptBuilder(dU.DEFAULT_TASK_PROMPT)
eval_collator = dU.MangaCollator(processor, process_vision_info, prompt_builder, train_mode=False, return_metadata=True)
sample_inputs, sample_metadata = eval_collator([validation_dataset[0]])
assert "labels" not in sample_inputs
assert sample_metadata[0]["target"] == validation_dataset[0]["target"]
print(f"Prompt tokens for first validation sequence: {sample_inputs['input_ids'].shape[1]}")
del sample_inputs, sample_metadata

Prompt tokens for first validation sequence: 1405


In [9]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the 4-bit VLM baseline")
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.bfloat16, quantization_config=quantization_config, device_map={"": 0})
base_model.eval()
print(f"Loaded base model on {next(base_model.parameters()).device}")

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Loaded base model on cuda:0


## Single-sample inference

Run this before the full eight-sequence evaluation to inspect one raw base-model response. Change `SAMPLE_INDEX` to examine another fixed validation sample.

In [10]:
SAMPLE_INDEX = 0
sample = validation_dataset[SAMPLE_INDEX]
sample_inputs, sample_metadata = eval_collator([sample])
device = next(base_model.parameters()).device
sample_inputs = {name: value.to(device) if torch.is_tensor(value) else value for name, value in sample_inputs.items()}

with torch.inference_mode():
    generated = base_model.generate(
        **sample_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        repetition_penalty=REPETITION_PENALTY,
    )

prompt_width = sample_inputs["input_ids"].shape[1]
prediction = processor.decode(generated[0, prompt_width:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
single_record = {"sequence_id": sample_metadata[0]["sequence_id"], "target": sample_metadata[0]["target"], "prediction": prediction}
single_metrics = calculate_metrics([single_record], minimum_similarity=MINIMUM_SIMILARITY)

print(f"SEQUENCE: {single_record['sequence_id']}")
print("\nTARGET:")
print(json.dumps(single_record["target"], ensure_ascii=False, indent=2))
print("\nRAW BASE-MODEL PREDICTION:")
print(prediction)
single_report = prediction_report(single_record, minimum_similarity=MINIMUM_SIMILARITY)
display_metric_summary(single_metrics, report=single_report)

SEQUENCE: seq_d8bc3c6ac1bb6017

TARGET:
[
  [
    {
      "speaker": "NARRATION",
      "text": "MARCH 4, 1985. JEROME PARK, THE BRONX, NYC"
    }
  ],
  [
    {
      "speaker": "character1",
      "text": "OH... GOD!"
    },
    {
      "speaker": "character2",
      "text": "M-MA'AM!?"
    },
    {
      "speaker": "character1",
      "text": "JERRY..."
    },
    {
      "speaker": "character1",
      "text": "JERRY!!!"
    },
    {
      "speaker": "character1",
      "text": "C-CALL... CALL AN AMBU... AMBU..."
    }
  ],
  [
    {
      "speaker": "character3",
      "text": "WHADD'YA THINK, JENKINS?"
    },
    {
      "speaker": "character4",
      "text": "SUICIDE?"
    },
    {
      "speaker": "character4",
      "text": "...PARDON ME."
    }
  ]
]

RAW BASE-MODEL PREDICTION:
[
  [
    {"speaker": "UNKNOWN", "text": "OH... GOD!"}
  ],
  [],
  [
    {"speaker": "UNKNOWN", "text": "WHAD'YA THINK, ARBU?"},
    {"speaker": "UNKNOWN", "text": "SUICIDE?"}
  ]
]


,Result,Value,What it tells us
0,Format valid?,Yes,1/1 outputs usable; check page-count diagnostics
1,Text error rate (lower is better),0.910,155 reference characters on valid outputs
2,Dialogue F1,0.333,"2 matched, 7 missing, 1 extra; 0 target lines ..."
3,Speaker accuracy,0.000,0/2 matched dialogues correct after character-...


In [11]:
baseline_metrics = evaluate_model(
    model=base_model,
    processor=processor,
    dataset=validation_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=MAX_NEW_TOKENS,
    minimum_similarity=MINIMUM_SIMILARITY,
    repetition_penalty=REPETITION_PENALTY,
    output_dir=OUTPUT_DIR,
)
print(f"Saved baseline outputs to {OUTPUT_DIR}")

Evaluated 1/8 sequences
Evaluated 2/8 sequences
Evaluated 3/8 sequences
Evaluated 4/8 sequences
Evaluated 5/8 sequences
Evaluated 6/8 sequences
Evaluated 7/8 sequences
Evaluated 8/8 sequences
Saved baseline outputs to C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\outputs\canonical_base_baseline_better_prompt


In [12]:
display_metric_summary(baseline_metrics)

records = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")
per_sequence = []
for record in records:
    metrics = calculate_metrics([record], minimum_similarity=MINIMUM_SIMILARITY)
    per_sequence.append({"sequence_id": record["sequence_id"], "format_valid": bool(metrics["schema_compliance_rate"]), "text_error_rate": metrics["cer_valid_outputs"], "failure_penalty_cer": metrics["cer_including_failures_as_deletions"], "missing_dialogues": metrics["support"]["missing_dialogues"] if metrics["cer_evaluated_sequences"] else None, "extra_dialogues": metrics["support"]["extra_dialogues"] if metrics["cer_evaluated_sequences"] else None, "speaker_accuracy": metrics["matched_speaker_accuracy"], "dialogue_f1": metrics["dialogue_text_match_f1"], "exact_match": metrics["sequence_exact_match"], "generated_tokens": record["generated_tokens"], "hit_max_new_tokens": record["hit_max_new_tokens"], "repetition_loop": record["suspected_repetition_loop"]})
review = pd.DataFrame(per_sequence).sort_values(["format_valid", "failure_penalty_cer"], ascending=[True, False]).reset_index(drop=True)
display(review.style.format(na_rep="N/A"))
_ = inspect_prediction(OUTPUT_DIR / "predictions.jsonl", sequence_id=review.iloc[0]["sequence_id"], minimum_similarity=MINIMUM_SIMILARITY)

,Result,Value,What it tells us
0,Format valid?,0.500,4/8 outputs usable; check page-count diagnostics
1,Text error rate (lower is better),1.219,1304 reference characters on valid outputs
2,Dialogue F1,0.254,"9 matched, 32 missing, 21 extra; 98 target lin..."
3,Speaker accuracy,0.111,1/9 matched dialogues correct after character-...


,sequence_id,format_valid,text_error_rate,failure_penalty_cer,missing_dialogues,extra_dialogues,speaker_accuracy,dialogue_f1,exact_match,generated_tokens,hit_max_new_tokens,repetition_loop
0,seq_535e0754d805105b,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
1,seq_dce8be23c3281228,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
2,seq_1c1d4f5e71a2f554,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
3,seq_098e260f231aa789,False,N/A,1.000000,N/A,N/A,N/A,N/A,0.000000,1024,True,False
4,seq_d7af6c64ebc22bbe,True,1.658291,1.658291,12.000000,9.000000,0.500000,0.160000,0.000000,258,False,False
5,seq_2032620aa4e4ac7f,True,1.250614,1.250614,10.000000,6.000000,0.000000,0.111111,0.000000,150,False,False
6,seq_d8bc3c6ac1bb6017,True,0.909677,0.909677,7.000000,1.000000,0.000000,0.333333,0.000000,66,False,False
7,seq_210721b0d576149f,True,0.811047,0.811047,3.000000,5.000000,0.000000,0.500000,0.000000,228,False,False


SEQUENCE
seq_535e0754d805105b

TARGET
[
  [
    {
      "speaker": "character1",
      "text": "SUICIDE, HUH..."
    },
    {
      "speaker": "character2",
      "text": "THE GUY LYING FACE-DOWN IS HIS BODYGUARD."
    },
    {
      "speaker": "character3",
      "text": "THIS IS THE THIRD CASE THIS YEAR IN OUR JURISDICTION ALONE."
    },
    {
      "speaker": "character3",
      "text": "'COURSE, THEIR HANDS WERE PRETTY BLOODY TO BEGIN WITH."
    },
    {
      "speaker": "character1",
      "text": "..."
    },
    {
      "speaker": "character2",
      "text": "OTHER PEOPLE'S LIVES DIDN'T MEAN A THING TO THEM..."
    },
    {
      "speaker": "character4",
      "text": "...BUT THEIR OWN, THEY HAD A PRETTY HIGH OPINION OF."
    },
    {
      "speaker": "character4",
      "text": "HARDLY THE SUICIDAL TYPE."
    },
    {
      "speaker": "character1",
      "text": "WHEN WAS THE FIRST ONE?"
    },
    {
      "speaker": "character5",
      "text": "I THINK CHARLIE HAS THE LIST..."

In [13]:
records = load_prediction_records(OUTPUT_DIR / "predictions.jsonl")

for record in records:
    print(
        record["sequence_id"],
        "tokens:", record["generated_tokens"],
        "hit limit:", record["hit_max_new_tokens"],
        "EOS:", record["reached_eos"],
    )

seq_d8bc3c6ac1bb6017 tokens: 66 hit limit: False EOS: True
seq_2032620aa4e4ac7f tokens: 150 hit limit: False EOS: True
seq_d7af6c64ebc22bbe tokens: 258 hit limit: False EOS: True
seq_535e0754d805105b tokens: 1024 hit limit: True EOS: False
seq_210721b0d576149f tokens: 228 hit limit: False EOS: True
seq_dce8be23c3281228 tokens: 1024 hit limit: True EOS: False
seq_1c1d4f5e71a2f554 tokens: 1024 hit limit: True EOS: False
seq_098e260f231aa789 tokens: 1024 hit limit: True EOS: False


## Official task score: primary experiment benchmark

Use the unmodified `TaskData/score.py` to judge text order and balanced text/speaker quality on the fixed holdout. Inspect invalid outputs and missing sequences before accepting an improvement.


In [ ]:
from Scripts.taskScore import score_saved_predictions

official_score = score_saved_predictions(OUTPUT_DIR / "predictions.jsonl", LABELS_PATH, validation_ids, OUTPUT_DIR / "task_score")
display(pd.Series(official_score["macro"], name="official_base_score").to_frame())
print("Missing or invalid sequences:", official_score["missing_sequences"])
